In [1]:
import pandas as pd
import numpy as np
np.random.seed( 42 )
from rdkit import Chem,DataStructs
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors, Lipinski ,QED,rdMolDescriptors,RDConfig
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
from mordred import Calculator, descriptors
import os
import sys
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
# now you can import sascore!
import sascorer
from tqdm import tqdm

In [9]:
!wget -O MOSES_Test_Smiles.csv https://media.githubusercontent.com/media/molecularsets/moses/master/data/test.csv

In [10]:
df = pd.read_csv('MOSES_Test_Smiles.csv')
df

In [ ]:
canonical_smiles = df['SMILES']
canonical_smiles

In [ ]:
def get_mol(smiles):
    moldata= []
    for elem in tqdm(smiles):
        mol=Chem.MolFromSmiles(elem)
        moldata.append(mol)
    return moldata
moldata = get_mol(canonical_smiles)

# Lipinski and RDKit descriptors

In [ ]:
def all_RDKit_descriptors(mol):
    descriptors = {}

    # Calculate all available descriptors
    for descriptor_name, descriptor_function in Descriptors.descList:
      #  if not descriptor_name.startswith('fr_'):
        try:
            descriptor_value = descriptor_function(mol)
            descriptors[descriptor_name] = descriptor_value
        except:
            descriptors[descriptor_name] = None

    return descriptors

In [ ]:
def desc_lipinski_RDKit(moldata, verbose=False):

    columnNames=["NumHDonors","NumHAcceptors","SAscore"]
    baseData= np.arange(1,1)
    i=0
    for mol in tqdm(moldata):
        desc_NumHDonors = Lipinski.NumHDonors(mol)
        desc_NumHAcceptors = Lipinski.NumHAcceptors(mol)

        SAscore = sascorer.calculateScore(mol)

        row = np.array([
                        desc_NumHDonors,
                        desc_NumHAcceptors,
                        SAscore,
                        ])
        descriptors = all_RDKit_descriptors(mol)
        for descriptor_name, descriptor_value in descriptors.items():
          row = np.append(row,descriptor_value)


        if(i==0):
            baseData=row
        else:
            baseData=np.vstack([baseData, row])
        i=i+1

    for descriptor_name, descriptor_value in tqdm(descriptors.items()):
      columnNames.append(descriptor_name)
    descriptors = pd.DataFrame(data=baseData,columns=columnNames)


    return descriptors

In [ ]:
df_descriptors = desc_lipinski_RDKit(moldata)
df_descriptors

In [ ]:
df_descriptors.to_csv('MOSES_Test_Lipinski and RDKit descriptors.csv', index=False)

# Mordred descriptor


In [ ]:
def mordred_descriptor(moldata, verbose=False):

    # Create a calculator object
    calc = Calculator(descriptors
                      , ignore_3D=True
                      )

    # Calculate descriptors for the molecules
    results = calc.pandas(moldata, nproc=os.cpu_count())

    return results

In [ ]:
df_mordred = mordred_descriptor(moldata)

In [ ]:
df_mordred

In [ ]:
df_mordred.to_csv('MOSES_Test_df_mordred descriptors.csv', index=False)

In [2]:
import rdkit

In [3]:
!pip install git+https://github.com/molecularsets/moses.git


  Cloning https://github.com/molecularsets/moses.git to c:\users\uas\appdata\local\temp\pip-req-build-j7lqmx_0


  ERROR: Error [WinError 2] The system cannot find the file specified while executing command git version
ERROR: Cannot find command 'git' - do you have 'git' installed and in your PATH?
